In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
train = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

**Plot Data**

In [ ]:
train

In [ ]:
def plot_images(data):
    fig, axes = plt.subplots(5, 5, figsize=(10, 10))
    axes = axes.ravel()

    for i in range(25):
        img = data.iloc[i, 1:].values.reshape(28, 28)  # Reshape the image
        label = data.iloc[i, 0]  # Get the label
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(label)
        axes[i].axis('off')
    
    plt.subplots_adjust(hspace=0.5)
    plt.show()


plot_images(train)

**Split data into train,test and validation**

In [ ]:
from sklearn.model_selection import train_test_split


x_img,val_img,x_label,val_label = train_test_split(train.iloc[:,1:],train.iloc[:,0],test_size=0.1,shuffle=True,random_state=43)

**Normalize Data 0-1**

In [ ]:
x_img/=255.0
val_img/=255.0
test/=255.0

**Converting data to tensor**

In [ ]:
x_img = tf.convert_to_tensor(x_img)
val_img = tf.convert_to_tensor(val_img)
x_label = tf.convert_to_tensor(x_label)
val_label = tf.convert_to_tensor(val_label)

**Reshaping 1d data to 28 by 28**

In [ ]:
x_img  = tf.reshape(x_img,(-1,28,28,1))
val_img  = tf.reshape(val_img,(-1,28,28,1))

In [ ]:
test = tf.reshape(test,(-1,28,28,1))

In [ ]:
x_img.shape

***Creating a simple CNN***

In [ ]:
from tensorflow.keras import models, layers 
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten

In [ ]:
inputs = tf.keras.Input(shape=(28,28,1))
new_layer1 = layers.Conv2D(32,kernel_size=6,activation='relu')(inputs)
new_layer2 = layers.Conv2D(16,kernel_size=6,activation='relu')(new_layer1)
flattened = layers.Flatten()(new_layer2)
outputs = layers.Dense(10,activation="softmax")(flattened)
naive_model = tf.keras.Model(inputs= inputs, outputs= outputs)

In [ ]:
naive_model.summary()

In [ ]:
naive_model.compile(loss=tf.keras.losses.sparse_categorical_crossentropy,
                   optimizer=tf.keras.optimizers.Adam(),
                   metrics=['accuracy'])

naive_history = naive_model.fit(x_img,x_label,batch_size=32,
                                epochs=10,
                               validation_data=(val_img,val_label))

In [ ]:
naive_prediction = naive_model.predict(test)

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
axes = axes.ravel()

for i in range(25):
    axes[i].imshow(test[i,:], cmap='gray')
    axes[i].set_title(f'{naive_prediction[i].argmax()}')
    axes[i].axis('off')

plt.subplots_adjust(hspace=0.5)
plt.show()


In [ ]:
naive_prediction = np.argmax(naive_prediction,axis = 1)

naive_prediction = pd.Series(naive_prediction,name="Label")

In [ ]:
naive_prediction

In [ ]:
submission = pd.concat([pd.Series(range(1,28001),name = "ImageId"),naive_prediction],axis = 1)

In [ ]:
submission.to_csv("first_time_CNN.csv",index=False)